# ANAADHI — Kaggle GPU Movie Shot Generator

GitHub-managed, Android-friendly notebook. Do not manually edit code unless asked.

**Current target:** `SC001_SH001` — Scene 001 exterior operation establishing shot.

Use a **Tesla T4** accelerator. P100 is intentionally rejected because the current Kaggle/PyTorch stack can fail on it.


In [ ]:
import sys, subprocess, importlib.util
required = ['diffusers', 'transformers', 'accelerate', 'safetensors', 'peft']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Required AI packages already available — skipping pip download.')

import torch
from pathlib import Path
from PIL import Image
from diffusers import AutoPipelineForText2Image
from IPython.display import display

assert torch.cuda.is_available(), 'GPU is not enabled. Kaggle Settings → Accelerator → GPU T4.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
print('CUDA:', torch.version.cuda)
if 'P100' in GPU_NAME.upper():
    raise RuntimeError('P100 is not supported by this notebook. Switch Kaggle Accelerator to GPU T4 and rerun.')


## Locked SC001_SH001 configuration — Pass 3

The previous candidates were rejected: 1101 had no cabin, 1102 made the cabin too small, 1103 became a tall watchtower, and 1104 drifted into concept-art/diagram imagery. This pass locks a broad low raised cabin, front stairs, surrounding operation and dark monsoon forest.


In [ ]:
SHOT_ID = 'SC001_SH001'
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
PROMPT = 'photoreal pre-dawn Western Ghats, broad rough-timber cabin raised only two metres above wet ground, front stairs, dark roof, black monsoon rain, giant roots, Indian police encircle cabin, rear security silhouettes, a few unmarked men behind, camouflaged medical vans under areca leaves, black-kite drones overhead, tense realistic anamorphic film still'
NEGATIVE_PROMPT = 'watchtower, treehouse, stilt tower, tiny hut, safari lodge, resort, thatched villa, open grassland, empty forest, daylight, sunny sky, sketch, concept art, diagram, labels, text, watermark, cyberpunk, American police, firefight, explosion, fantasy, blurry'
SEEDS = [1103, 1201, 1202, 1203, 1204, 1205]
STEPS = 36
GUIDANCE = 7.5
GEN_WIDTH = 1344
GEN_HEIGHT = 640
MASTER_WIDTH = 3840
MASTER_HEIGHT = 1608
OUTPUT_DIR = Path('/kaggle/working/anaadhi_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Shot:', SHOT_ID)
print('Candidate seeds:', SEEDS)


## Load SDXL and verify prompt length

This pass explicitly checks both SDXL tokenizers and stops before generation if either prompt exceeds the 77-token CLIP limit.


In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()

def token_len(tokenizer, text):
    return len(tokenizer(text, truncation=False, add_special_tokens=True)['input_ids'])

for label, tok in [('CLIP-1', pipe.tokenizer), ('CLIP-2', pipe.tokenizer_2)]:
    p_len = token_len(tok, PROMPT)
    n_len = token_len(tok, NEGATIVE_PROMPT)
    print(label, 'prompt tokens:', p_len, '| negative tokens:', n_len)
    if p_len > 77 or n_len > 77:
        raise RuntimeError(f'{label} prompt exceeds 77 tokens. GitHub prompt must be shortened before rendering.')

print('Model ready:', MODEL_ID)


## Generate six SC001_SH001 candidates

Each candidate is centre-cropped to the practical 2.39:1 ratio matching the 3840×1608 ANAADHI movie master.


In [ ]:
target_ratio = MASTER_WIDTH / MASTER_HEIGHT
saved = []
for seed in SEEDS:
    print(f'Generating candidate seed {seed}...')
    generator = torch.Generator(device='cpu').manual_seed(seed)
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=GEN_WIDTH,
        height=GEN_HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=generator,
    ).images[0]

    new_h = round(image.width / target_ratio)
    if new_h <= image.height:
        y0 = (image.height - new_h) // 2
        scoped = image.crop((0, y0, image.width, y0 + new_h))
    else:
        new_w = round(image.height * target_ratio)
        x0 = (image.width - new_w) // 2
        scoped = image.crop((x0, 0, x0 + new_w, image.height))

    out_path = OUTPUT_DIR / f'{SHOT_ID}_seed{seed}.png'
    scoped.save(out_path)
    saved.append(out_path)
    print('Saved:', out_path, 'size:', scoped.size, 'ratio:', round(scoped.width / scoped.height, 4))
    display(scoped)

print('Finished', len(saved), 'candidates.')


## Next

Send the six candidate screenshots to ChatGPT. Do not edit Python on the phone. ChatGPT will approve one or revise GitHub for the next pass.
